# nb-05: Cross-Reaction Consistency Analysis

Find the same (reactant, product) pair across multiple reactions and check
whether RDT maps it consistently in each.

In [ ]:
from pathlib import Path
from consistency import (
    load_all_reactions,
    find_inconsistencies,
    identity_exact,
)

## 1. Load all reactions

In [ ]:
ARACORE = Path(".")
load_result = load_all_reactions(ARACORE / "reaction_intermediates.zip")
print(f"Loaded: {len(load_result.reactions)} reactions")
print(f"Skipped: {len(load_result.skipped)}")
for name, reason in load_result.skipped:
    print(f"  {name}: {reason}")

## 2. Run consistency analysis

In [ ]:
result = find_inconsistencies(load_result.reactions)

## 3. Label violations

In [ ]:
print(f"Label violations: {len(result.label_violations)}")
print(f"Excluded species: {result.excluded_species}")
for lv in result.label_violations:
    print(f"  {lv.species_id}:")
    for rxn, inchi in lv.inchi_strings.items():
        print(f"    {rxn}: {inchi}")

## 4. Inconsistency summary

In [ ]:
print(f"Inconsistent pairs: {len(result.inconsistencies)}")
for inc in result.inconsistencies:
    print(f"\n  {inc.from_species} -> {inc.to_species} ({len(inc.classes)} classes):")
    for cls in inc.classes:
        print(f"    {cls.reactions}: {len(cls.mapping)} atom pairs")

## 5. Drill into one inconsistent pair

In [ ]:
if result.inconsistencies:
    inc = result.inconsistencies[0]
    print(f"Pair: {inc.from_species} -> {inc.to_species}")
    for i, cls in enumerate(inc.classes):
        print(f"\nClass {i}: reactions={cls.reactions}")
        for pair in sorted(cls.mapping):
            print(f"  {pair[0]} -> {pair[1]}")
    if len(inc.classes) == 2:
        only_in_a = inc.classes[0].mapping - inc.classes[1].mapping
        only_in_b = inc.classes[1].mapping - inc.classes[0].mapping
        if only_in_a or only_in_b:
            print(f"\nDifferences:")
            print(f"  Only in {inc.classes[0].reactions}: {sorted(only_in_a)}")
            print(f"  Only in {inc.classes[1].reactions}: {sorted(only_in_b)}")
else:
    print("No inconsistencies found.")